In [4]:
"""
Graph Sensitivity Experiment — Causal SHAP on Yield Regression
===============================================================

Tests 11 causal graph variants against the true DGP graph and measures
attribution quality via Pearson r, Spearman r, Cosine similarity, RMSE,
and MAE vs ground-truth gamma_i.

Graph categories
----------------
G00          — True star DAG (no inter-feature edges, all 4 -> Yield)
G01–G08      — Spurious inter-feature edges added (1–4 edges)
G09–G10      — One feature disconnected from Yield (missing edge)

Usage
-----
    python causal_shap_graph_sensitivity.py

Outputs (written to working directory)
---------------------------------------
    graph_sensitivity_results.csv     — metrics for all 11 graphs
    graph_sensitivity_bar.png         — Pearson r bar chart
    graph_sensitivity_heatmap.png     — all metrics heatmap
"""

import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from typing import Dict, List, Set, Tuple

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
import networkx as nx

try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except ImportError:
    from sklearn.ensemble import RandomForestRegressor
    HAVE_XGB = False

#  CONFIG  - adjust here to trade speed for accuracy
CONFIG = {
    # Data
    'DATA_PATH':   '../datasets/yield_regression_preprocessed.npz',

    # Causal SHAP sampling
    'MAX_PERM':    200,    # Shapley permutations per instance
    'M_RESAMPLES': 64,     # MC samples per vc(S) call
    'GLOBAL_K':    30,     # test instances per graph

    # Reproducibility
    'RANDOM_SEED': 42,
}

FEATURE_NAMES = ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']

TRANSLOG_COEFFS = {
    'betaN':  0.35,  'betaI':  0.50,  'betaP':  0.25,  'betaH':  0.05,
    'betaNN': -0.02, 'betaII': -0.03, 'betaPP': -0.01, 'betaHH': -0.005,
    'betaNI': 0.04,  'betaNP': 0.02,  'betaNH': 0.01,
    'betaIP': 0.03,  'betaIH': 0.01,  'betaPH': 0.005,
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])


#  GAMMA COMPUTATION  (analytic elasticities - unchanged)
def compute_gamma(X_train: pd.DataFrame) -> Dict[str, float]:
    """Derive gamma_i from translog structural equation via elasticities."""
    c  = TRANSLOG_COEFFS
    lm = {f: float(np.log(X_train[f]).mean()) for f in FEATURE_NAMES}
    lnN, lnI, lnP, lnH = (lm['Nitrogen'], lm['Irrigation'],
                           lm['Precipitation'], lm['Herbicide'])
    elasticities = {
        'Nitrogen':
            c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI
            + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':
            c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN
            + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation':
            c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN
            + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':
            c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN
            + c['betaIH']*lnI + c['betaPH']*lnP,
    }
    total = sum(elasticities.values())
    return {f: v / total for f, v in elasticities.items()}


# Build dag from graph spec
def build_dag(inter_edges: List[Tuple[str, str]]) -> nx.DiGraph:
    """Build feature-only DAG (no Yield node) from inter-feature edge list."""
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for (u, v) in inter_edges:
        G.add_edge(u, v)
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError(f"Cycle detected in inter_edges: {inter_edges}")
    return G


#  LINEAR-GAUSSIAN SEM (Optimized Batch Sampling)
class LinearMechanisms:
    """
    Fit a linear-Gaussian SEM on the known feature DAG.
    Used to sample X_{not S} | do(X_S = x_S) during Causal SHAP.
    """

    def __init__(self, dag: nx.DiGraph):
        self.dag  = dag
        self.mech: Dict[str, dict] = {}
        self.topo: List[str] = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        for var in FEATURE_NAMES:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def sample_intervention_batch(self, S: Set[str], x_obs: pd.Series, M: int) -> np.ndarray:
        """Draw M samples from P(X_{not S} | do(X_S = x_S)) in a vectorized manner."""
        # Pre-allocate array: rows=samples, cols=features ordered by FEATURE_NAMES
        X_samples = np.empty((M, len(FEATURE_NAMES)))
        feat_to_idx = {f: i for i, f in enumerate(FEATURE_NAMES)}

        # Track simulated feature states across topological order
        simulated = {}

        for var in self.topo:
            v_idx = feat_to_idx[var]
            if var in S:
                val = float(x_obs[var])
                X_samples[:, v_idx] = val
                simulated[var] = np.full(M, val)
                continue

            m = self.mech[var]
            sigma = m['sigma']

            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                # Accumulate parents matrix: shape (M, n_parents)
                p_matrix = np.empty((M, len(m['parents'])))
                for p_idx, p in enumerate(m['parents']):
                    if p in simulated:
                        p_matrix[:, p_idx] = simulated[p]
                    else:
                        p_matrix[:, p_idx] = float(x_obs[p])
                mu = m['intercept'] + p_matrix @ m['coef']

            if sigma < 1e-8:
                vals = mu
            else:
                vals = rng.normal(mu, sigma, size=M)

            X_samples[:, v_idx] = vals
            simulated[var] = vals

        return X_samples


#  CAUSAL SHAP - REGRESSION (Vectorized)
class CausalSHAPRegressor:
    """
    Causal SHAP for regression with a specified causal graph and gamma_i.
    """

    def __init__(self, model, dag: nx.DiGraph,
                 gamma: Dict[str, float],
                 missing_yield: List[str],
                 X_train: pd.DataFrame):
        self.model    = model
        self.features = list(X_train.columns)
        self._Ef      = None

        # Zero out gamma for features disconnected from Yield, then renormalise
        eff = {f: (0.0 if f in missing_yield else gamma[f]) for f in self.features}
        tot = sum(eff.values())
        self.gamma = {f: v / tot for f, v in eff.items()} if tot > 1e-12 else eff

        # Fit SEM on feature DAG
        self.sem = LinearMechanisms(dag).fit(X_train)

        # Cache E[f(X)]
        self._Ef = float(model.predict(X_train.values).mean())

    def _vc_batch(self, S: Set[str], x: pd.Series, M: int) -> float:
        """Vectorized Monte Carlo estimate of E[f(X) | do(X_S = x_S)]."""
        # Obtain all M samples concurrently from structural equation models
        X_mat = self.sem.sample_intervention_batch(S, x, M)
        # Execute batch inference across the entire sample matrix using clean arrays
        preds = self.model.predict(X_mat)
        return float(np.mean(preds))

    def explain_instance(self, x: pd.Series,
                         n_perms: int, M: int) -> pd.Series:
        """Causal SHAP for one instance (Eq. 10 + Eq. 11 normalisation)."""
        x_row = x.values.reshape(1, -1)
        fx = float(self.model.predict(x_row)[0])

        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm   = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc_batch(S, x, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc_batch(S_next, x, M)
                phi[self.features.index(feat)] += (
                    self.gamma[feat] * (v_curr - v_prev))    # Eq. 10
                v_prev, S = v_curr, S_next
        phi /= n_perms

        # Eq. 11: rescale so Σ φ_i = f(x) - E[f(X)]
        s = float(np.sum(phi))
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)

        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame,
                       k: int, n_perms: int, M: int) -> pd.Series:
        """
        Global Causal SHAP: mean absolute attribution over k test instances.
        """
        k   = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        phi_list = []
        for t, i in enumerate(idx, 1):
            if t % 5 == 0:
                print(f"      instance {t}/{k} ...", flush=True)
            phi_list.append(self.explain_instance(X_test.iloc[i], n_perms, M))
        phi_mat = pd.concat(phi_list, axis=1)
        return phi_mat.abs().mean(axis=1).rename('Causal_SHAP_AbsMean')


# Evaluation metrics
def compute_metrics(gamma: Dict[str, float],
                    global_abs: pd.Series) -> Dict[str, float]:
    """
    Compare normalised Causal SHAP AbsMean against GT gamma_i.
    """
    gt_vec   = np.array([gamma[f]            for f in FEATURE_NAMES])
    raw      = np.array([global_abs.get(f,0) for f in FEATURE_NAMES])
    shap_vec = raw / raw.sum() if raw.sum() > 1e-12 else raw

    pearson_r,  _ = pearsonr(shap_vec,  gt_vec)
    spearman_r, _ = spearmanr(shap_vec, gt_vec)
    cosine        = float(np.dot(shap_vec, gt_vec) /
                          (np.linalg.norm(shap_vec) * np.linalg.norm(gt_vec) + 1e-12))
    rmse          = float(np.sqrt(np.mean((shap_vec - gt_vec) ** 2)))
    mae           = float(np.mean(np.abs(shap_vec - gt_vec)))

    return {
        'Pearson_r':  round(float(pearson_r),  4),
        'Spearman_r': round(float(spearman_r), 4),
        'Cosine':     round(cosine,            4),
        'RMSE':       round(rmse,              4),
        'MAE':        round(mae,               4),
    }


# Plots
def plot_bar_chart(results_df: pd.DataFrame, output_path: str):
    """Horizontal bar chart of Pearson r for all 11 graphs."""
    fig, ax = plt.subplots(figsize=(10, 7))

    true_pr = results_df.loc[results_df['Label'] == 'G00: True graph', 'Pearson_r'].iloc[0]
    colors  = [CAT_COLORS[c] for c in results_df['Category']]

    bars = ax.barh(results_df['Label'], results_df['Pearson_r'],
                   color=colors, edgecolor='white', height=0.65)

    for bar, val in zip(bars, results_df['Pearson_r']):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', ha='left', fontsize=9.5)

    ax.axvline(true_pr, color='#1D9E75', linewidth=2, linestyle='--', alpha=0.8)
    ax.text(true_pr + 0.005, -0.8, f'True graph ({true_pr:.4f})',
            color='#1D9E75', fontsize=9, va='top')

    ax.set_xlabel('Pearson r  (normalised Causal SHAP AbsMean vs GT γᵢ)', fontsize=11)
    ax.set_xlim(0, 1.12)
    ax.set_title('Graph Sensitivity: Pearson r Across 11 Causal Graph Variants\n'
                 'Yield Regression — XGBRegressor — GT γ from translog elasticities',
                 fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

    from matplotlib.patches import Patch
    legend_handles = [
        Patch(color=CAT_COLORS['baseline'],   label='True (baseline)'),
        Patch(color=CAT_COLORS['extra_edge'], label='Spurious inter-feature edge'),
        Patch(color=CAT_COLORS['chain'],      label='Chain structure'),
        Patch(color=CAT_COLORS['missing'],    label='Missing Yield edge'),
    ]
    ax.legend(handles=legend_handles, loc='lower right', fontsize=9, frameon=False)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved {output_path}")


def plot_heatmap(results_df: pd.DataFrame, output_path: str):
    """Heatmap of all 4 metrics × 11 graphs."""
    metrics = ['Pearson_r', 'Spearman_r', 'Cosine', 'RMSE']
    data    = results_df[metrics].values.T

    fig, ax = plt.subplots(figsize=(13, 4))
    im = ax.imshow(data, aspect='auto', cmap='RdYlGn',
                   vmin=data.min(), vmax=1.0)

    ax.set_xticks(range(len(results_df)))
    ax.set_xticklabels(results_df['Label'], rotation=40, ha='right', fontsize=9)
    ax.set_yticks(range(len(metrics)))
    ax.set_yticklabels(metrics, fontsize=10)

    for i in range(len(metrics)):
        for j in range(len(results_df)):
            val = data[i, j]
            text_col = 'white' if (metrics[i] == 'RMSE' and val > 0.2) else 'black'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=8.5, color=text_col, fontweight='bold')

    plt.colorbar(im, ax=ax, label='Score (higher = better, except RMSE)')
    ax.set_title('Graph Sensitivity — All Metrics Heatmap\n'
                 '(RMSE: lower = better; others: higher = better)',
                 fontsize=11, fontweight='bold')
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved {output_path}")


# Graph definitions & config loading
GRAPH_VARIANTS = [
    dict(label='G00: True graph', inter_edges=[], missing_yield=[], category='baseline', description='True DGP: star DAG, no inter-feature edges, all 4 → Yield'),
    dict(label='G01: +N→I', inter_edges=[('Nitrogen', 'Irrigation')], missing_yield=[], category='extra_edge', description='Add 1 wrong edge: Nitrogen → Irrigation'),
    dict(label='G02: +I→N', inter_edges=[('Irrigation', 'Nitrogen')], missing_yield=[], category='extra_edge', description='Add 1 wrong edge: Irrigation → Nitrogen (reversed)'),
    dict(label='G03: +N→P', inter_edges=[('Nitrogen', 'Precipitation')], missing_yield=[], category='extra_edge', description='Add 1 wrong edge: Nitrogen → Precipitation'),
    dict(label='G04: +P→H', inter_edges=[('Precipitation', 'Herbicide')], missing_yield=[], category='extra_edge', description='Add 1 wrong edge: Precipitation → Herbicide'),
    dict(label='G05: +N→I, +P→H', inter_edges=[('Nitrogen', 'Irrigation'), ('Precipitation', 'Herbicide')], missing_yield=[], category='extra_edge', description='Add 2 wrong edges: N→I and P→H'),
    dict(label='G06: N→I→P chain', inter_edges=[('Nitrogen', 'Irrigation'), ('Irrigation', 'Precipitation')], missing_yield=[], category='chain', description='Add 3-node chain: N→I→P'),
    dict(label='G07: Full chain N→I→P→H', inter_edges=[('Nitrogen', 'Irrigation'), ('Irrigation', 'Precipitation'), ('Precipitation', 'Herbicide')], missing_yield=[], category='chain', description='Full 4-node chain: N→I→P→H (all wrong)'),
    dict(label='G08: Dense (4 wrong edges)', inter_edges=[('Nitrogen', 'Irrigation'), ('Nitrogen', 'Precipitation'), ('Irrigation', 'Precipitation'), ('Irrigation', 'Herbicide')], missing_yield=[], category='extra_edge', description='Dense wrong graph: 4 spurious inter-feature edges'),
    dict(label='G09: No Irrigation→Yield', inter_edges=[], missing_yield=['Irrigation'], category='missing', description='Irrigation disconnected from Yield (feature present, edge absent)'),
    dict(label='G10: No Nitrogen→Yield', inter_edges=[], missing_yield=['Nitrogen'], category='missing', description='Nitrogen disconnected from Yield (feature present, edge absent)'),
]

CAT_COLORS = {'baseline': '#1D9E75', 'extra_edge': '#378ADD', 'chain': '#7F77DD', 'missing': '#D85A30'}


def main():
    print("\n" + "=" * 70)
    print("GRAPH SENSITIVITY EXPERIMENT — Causal SHAP on Yield Regression")
    print("=" * 70)
    print(f"  Graphs to test : {len(GRAPH_VARIANTS)}")
    print(f"  MAX_PERM       : {CONFIG['MAX_PERM']}")
    print(f"  M_RESAMPLES    : {CONFIG['M_RESAMPLES']}")
    print(f"  GLOBAL_K       : {CONFIG['GLOBAL_K']}")
    print()

    try:
        data = np.load(CONFIG['DATA_PATH'], allow_pickle=True)
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['DATA_PATH']}' not found."); return

    X_train = pd.DataFrame(data['X_train'], columns=FEATURE_NAMES)
    X_test  = pd.DataFrame(data['X_test'],  columns=FEATURE_NAMES)
    y_train = data['y_train'].astype(float)
    y_test  = data['y_test'].astype(float)
    print(f"Data: X_train={X_train.shape}  X_test={X_test.shape}")

    gamma = compute_gamma(X_train)
    print("\nGT gamma_i (analytic translog elasticities):")
    for f, v in gamma.items():
        print(f"  {f:<18}  γ = {v:.4f}")

    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_NAMES)
    Xte = pd.DataFrame(scaler.transform(X_test),      columns=FEATURE_NAMES)

    print("\nTraining model ...")
    if HAVE_XGB:
        model = XGBRegressor(
            n_estimators=160, max_depth=5, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9,
            random_state=CONFIG['RANDOM_SEED'], verbosity=0,
            n_jobs=1,
            tree_method='hist'
        )
        model_name = "XGBRegressor"
    else:
        model = RandomForestRegressor(
            n_estimators=160, max_depth=6,
            random_state=CONFIG['RANDOM_SEED'], n_jobs=1)
        model_name = "RandomForestRegressor"

    model.fit(Xtr.values, y_train)
    r2 = r2_score(y_test, model.predict(Xte.values))
    print(f"  {model_name}  R² = {r2:.4f}")

    all_results = []
    t_total = time.time()

    for i, g in enumerate(GRAPH_VARIANTS):
        print(f"\n[{i+1:02d}/{len(GRAPH_VARIANTS)}] {g['label']}")
        print(f"         {g['description']}")
        t0 = time.time()

        dag = build_dag(g['inter_edges'])
        cshap = CausalSHAPRegressor(
            model         = model,
            dag           = dag,
            gamma         = gamma,
            missing_yield = g['missing_yield'],
            X_train       = Xtr,
        )

        global_abs = cshap.explain_global(
            Xte,
            k       = CONFIG['GLOBAL_K'],
            n_perms = CONFIG['MAX_PERM'],
            M       = CONFIG['M_RESAMPLES'],
        )

        m = compute_metrics(gamma, global_abs)
        elapsed = time.time() - t0

        print(f"         Pearson={m['Pearson_r']}  Spearman={m['Spearman_r']}"
              f"  Cosine={m['Cosine']}  RMSE={m['RMSE']}  ({elapsed:.1f}s)")

        all_results.append({
            'Label':           g['label'],
            'Category':        g['category'],
            'Description':     g['description'],
            'N_wrong_edges':   len(g['inter_edges']),
            'N_missing_yield': len(g['missing_yield']),
            **m,
            'Time_s':          round(elapsed, 1),
        })

    total_time = time.time() - t_total
    print(f"\nTotal elapsed: {total_time/60:.1f} min")

    df = pd.DataFrame(all_results)
    df.to_csv('graph_sensitivity_results.csv', index=False)
    print("\n" + "=" * 70)
    print("FINAL RESULTS")
    print("=" * 70)
    print(df[['Label', 'Pearson_r', 'Spearman_r', 'Cosine', 'RMSE', 'MAE']].to_string(index=False))

    plot_bar_chart(df, 'graph_sensitivity_bar.png')
    plot_heatmap(df,   'graph_sensitivity_heatmap.png')

    true_row  = df[df['Label'] == 'G00: True graph'].iloc[0]
    best_pr   = df.loc[df['Pearson_r'].idxmax()]
    worst_pr  = df.loc[df['Pearson_r'].idxmin()]
    miss_rows = df[df['N_missing_yield'] > 0]

    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(f"  True graph Pearson r     : {true_row['Pearson_r']}")
    print(f"  Best Pearson r           : {best_pr['Pearson_r']}  ({best_pr['Label']})")
    print(f"  Worst Pearson r          : {worst_pr['Pearson_r']} ({worst_pr['Label']})")
    print(f"  True graph rank (Pearson): {df['Pearson_r'].rank(ascending=False).loc[df['Label']=='G00: True graph'].iloc[0]:.0f} / {len(df)}")
    print()


if __name__ == '__main__':
    main()


GRAPH SENSITIVITY EXPERIMENT — Causal SHAP on Yield Regression
  Graphs to test : 11
  MAX_PERM       : 200
  M_RESAMPLES    : 64
  GLOBAL_K       : 30

Data: X_train=(800, 4)  X_test=(200, 4)

GT gamma_i (analytic translog elasticities):
  Nitrogen            γ = 0.3118
  Irrigation          γ = 0.3727
  Precipitation       γ = 0.2457
  Herbicide           γ = 0.0697

Training model ...
  XGBRegressor  R² = 0.9336

[01/11] G00: True graph
         True DGP: star DAG, no inter-feature edges, all 4 → Yield
      instance 5/30 ...
      instance 10/30 ...
      instance 15/30 ...
      instance 20/30 ...
      instance 25/30 ...
      instance 30/30 ...
         Pearson=0.8612  Spearman=0.8  Cosine=0.9275  RMSE=0.1247  (34.2s)

[02/11] G01: +N→I
         Add 1 wrong edge: Nitrogen → Irrigation
      instance 5/30 ...
      instance 10/30 ...
      instance 15/30 ...
      instance 20/30 ...
      instance 25/30 ...
      instance 30/30 ...
         Pearson=0.8485  Spearman=0.8  Cosine=0

In [5]:
"""
Graph Sensitivity Experiment — Causal SHAP on Yield Regression (High-Fidelity)
=============================================================================

Tests 11 causal graph variants against the true DGP graph and measures
attribution quality vs ground-truth gamma_i.

Upgraded to log exact feature attributions and use higher sampling thresholds.
"""

import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from typing import Dict, List, Set, Tuple

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
import networkx as nx

try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except ImportError:
    from sklearn.ensemble import RandomForestRegressor
    HAVE_XGB = False

#  CONFIG  - Upgraded for higher fidelity and precision
CONFIG = {
    # Data
    'DATA_PATH':   '../datasets/yield_regression_preprocessed.npz',

    # Causal SHAP sampling (Increased parameters for stability)
    'MAX_PERM':    500,    # Scaled from 200 -> More robust Shapley permutations
    'M_RESAMPLES': 128,    # Scaled from 64  -> Cleaner conditional SEM integration
    'GLOBAL_K':    100,    # Scaled from 30  -> Broader test evaluation pool

    # Reproducibility
    'RANDOM_SEED': 42,
}

FEATURE_NAMES = ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']

TRANSLOG_COEFFS = {
    'betaN':  0.35,  'betaI':  0.50,  'betaP':  0.25,  'betaH':  0.05,
    'betaNN': -0.02, 'betaII': -0.03, 'betaPP': -0.01, 'betaHH': -0.005,
    'betaNI': 0.04,  'betaNP': 0.02,  'betaNH': 0.01,
    'betaIP': 0.03,  'betaIH': 0.01,  'betaPH': 0.005,
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])


def compute_gamma(X_train: pd.DataFrame) -> Dict[str, float]:
    """Derive gamma_i from translog structural equation via elasticities."""
    c  = TRANSLOG_COEFFS
    lm = {f: float(np.log(X_train[f]).mean()) for f in FEATURE_NAMES}
    lnN, lnI, lnP, lnH = (lm['Nitrogen'], lm['Irrigation'],
                           lm['Precipitation'], lm['Herbicide'])
    elasticities = {
        'Nitrogen':
            c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI
            + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':
            c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN
            + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation':
            c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN
            + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':
            c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN
            + c['betaIH']*lnI + c['betaPH']*lnP,
    }
    total = sum(elasticities.values())
    return {f: v / total for f, v in elasticities.items()}


def build_dag(inter_edges: List[Tuple[str, str]]) -> nx.DiGraph:
    """Build feature-only DAG (no Yield node) from inter-feature edge list."""
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for (u, v) in inter_edges:
        G.add_edge(u, v)
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError(f"Cycle detected in inter_edges: {inter_edges}")
    return G


class LinearMechanisms:
    """Fit a linear-Gaussian SEM on the known feature DAG."""
    def __init__(self, dag: nx.DiGraph):
        self.dag  = dag
        self.mech: Dict[str, dict] = {}
        self.topo: List[str] = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        for var in FEATURE_NAMES:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def sample_intervention_batch(self, S: Set[str], x_obs: pd.Series, M: int) -> np.ndarray:
        X_samples = np.empty((M, len(FEATURE_NAMES)))
        feat_to_idx = {f: i for i, f in enumerate(FEATURE_NAMES)}
        simulated = {}

        for var in self.topo:
            v_idx = feat_to_idx[var]
            if var in S:
                val = float(x_obs[var])
                X_samples[:, v_idx] = val
                simulated[var] = np.full(M, val)
                continue

            m = self.mech[var]
            sigma = m['sigma']

            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                p_matrix = np.empty((M, len(m['parents'])))
                for p_idx, p in enumerate(m['parents']):
                    if p in simulated:
                        p_matrix[:, p_idx] = simulated[p]
                    else:
                        p_matrix[:, p_idx] = float(x_obs[p])
                mu = m['intercept'] + p_matrix @ m['coef']

            if sigma < 1e-8:
                vals = mu
            else:
                vals = rng.normal(mu, sigma, size=M)

            X_samples[:, v_idx] = vals
            simulated[var] = vals

        return X_samples


class CausalSHAPRegressor:
    """Causal SHAP calculation suite."""
    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float], missing_yield: List[str], X_train: pd.DataFrame):
        self.model    = model
        self.features = list(X_train.columns)
        self._Ef      = None

        eff = {f: (0.0 if f in missing_yield else gamma[f]) for f in self.features}
        tot = sum(eff.values())
        self.gamma = {f: v / tot for f, v in eff.items()} if tot > 1e-12 else eff
        self.sem = LinearMechanisms(dag).fit(X_train)
        self._Ef = float(model.predict(X_train.values).mean())

    def _vc_batch(self, S: Set[str], x: pd.Series, M: int) -> float:
        X_mat = self.sem.sample_intervention_batch(S, x, M)
        preds = self.model.predict(X_mat)
        return float(np.mean(preds))

    def explain_instance(self, x: pd.Series, n_perms: int, M: int) -> pd.Series:
        x_row = x.values.reshape(1, -1)
        fx = float(self.model.predict(x_row)[0])

        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm   = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc_batch(S, x, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc_batch(S_next, x, M)
                phi[self.features.index(feat)] += (self.gamma[feat] * (v_curr - v_prev))
                v_prev, S = v_curr, S_next
        phi /= n_perms

        s = float(np.sum(phi))
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)

        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame, k: int, n_perms: int, M: int) -> pd.Series:
        k   = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        phi_list = []
        for t, i in enumerate(idx, 1):
            if t % 10 == 0:
                print(f"      instance {t}/{k} ...", flush=True)
            phi_list.append(self.explain_instance(X_test.iloc[i], n_perms, M))
        phi_mat = pd.concat(phi_list, axis=1)
        return phi_mat.abs().mean(axis=1).rename('Causal_SHAP_AbsMean')


def compute_metrics(gamma: Dict[str, float], global_abs: pd.Series) -> Tuple[Dict[str, float], Dict[str, float]]:
    """Compares metrics and extracts both raw and relative assigned attributions."""
    gt_vec   = np.array([gamma[f]            for f in FEATURE_NAMES])
    raw      = np.array([global_abs.get(f,0) for f in FEATURE_NAMES])
    shap_vec = raw / raw.sum() if raw.sum() > 1e-12 else raw

    pearson_r,  _ = pearsonr(shap_vec,  gt_vec)
    spearman_r, _ = spearmanr(shap_vec, gt_vec)
    cosine        = float(np.dot(shap_vec, gt_vec) / (np.linalg.norm(shap_vec) * np.linalg.norm(gt_vec) + 1e-12))
    rmse          = float(np.sqrt(np.mean((shap_vec - gt_vec) ** 2)))
    mae           = float(np.mean(np.abs(shap_vec - gt_vec)))

    metrics = {
        'Pearson_r':  round(float(pearson_r),  4),
        'Spearman_r': round(float(spearman_r), 4),
        'Cosine':     round(cosine,            4),
        'RMSE':       round(rmse,              4),
        'MAE':        round(mae,               4),
    }

    # Save the relative attribution share assigned to each feature for tracking shifts
    attributions = {f"Attr_{f}": round(shap_vec[i], 4) for i, f in enumerate(FEATURE_NAMES)}

    return metrics, attributions


# Plots
def plot_bar_chart(results_df: pd.DataFrame, output_path: str):
    fig, ax = plt.subplots(figsize=(10, 7))
    true_pr = results_df.loc[results_df['Label'] == 'G00: True graph', 'Pearson_r'].iloc[0]
    colors  = [CAT_COLORS[c] for c in results_df['Category']]
    bars = ax.barh(results_df['Label'], results_df['Pearson_r'], color=colors, edgecolor='white', height=0.65)

    for bar, val in zip(bars, results_df['Pearson_r']):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2, f'{val:.4f}', va='center', ha='left', fontsize=9.5)

    ax.axvline(true_pr, color='#1D9E75', linewidth=2, linestyle='--', alpha=0.8)
    ax.set_xlabel('Pearson r  (normalised Causal SHAP AbsMean vs GT γᵢ)', fontsize=11)
    ax.set_xlim(0, 1.12)
    ax.set_title('Graph Sensitivity (High-Res): Pearson r Across 11 Variants', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)


def plot_heatmap(results_df: pd.DataFrame, output_path: str):
    metrics = ['Pearson_r', 'Spearman_r', 'Cosine', 'RMSE']
    data    = results_df[metrics].values.T
    fig, ax = plt.subplots(figsize=(13, 4))
    im = ax.imshow(data, aspect='auto', cmap='RdYlGn', vmin=data.min(), vmax=1.0)
    ax.set_xticks(range(len(results_df)))
    ax.set_xticklabels(results_df['Label'], rotation=40, ha='right', fontsize=9)
    ax.set_yticks(range(len(metrics)))
    ax.set_yticklabels(metrics, fontsize=10)

    for i in range(len(metrics)):
        for j in range(len(results_df)):
            val = data[i, j]
            text_col = 'white' if (metrics[i] == 'RMSE' and val > 0.2) else 'black'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8.5, color=text_col, fontweight='bold')

    plt.colorbar(im, ax=ax)
    ax.set_title('Graph Sensitivity High-Res Heatmap', fontsize=11, fontweight='bold')
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)


GRAPH_VARIANTS = [
    dict(label='G00: True graph', inter_edges=[], missing_yield=[], category='baseline', description='True DGP: star DAG, no inter-feature edges, all 4 → Yield'),
    dict(label='G01: +N→I', inter_edges=[('Nitrogen', 'Irrigation')], missing_yield=[], category='extra_edge', description='Add 1 wrong edge: Nitrogen → Irrigation'),
    dict(label='G02: +I→N', inter_edges=[('Irrigation', 'Nitrogen')], missing_yield=[], category='extra_edge', description='Add 1 wrong edge: Irrigation → Nitrogen (reversed)'),
    dict(label='G03: +N→P', inter_edges=[('Nitrogen', 'Precipitation')], missing_yield=[], category='extra_edge', description='Add 1 wrong edge: Nitrogen → Precipitation'),
    dict(label='G04: +P→H', inter_edges=[('Precipitation', 'Herbicide')], missing_yield=[], category='extra_edge', description='Add 1 wrong edge: Precipitation → Herbicide'),
    dict(label='G05: +N→I, +P→H', inter_edges=[('Nitrogen', 'Irrigation'), ('Precipitation', 'Herbicide')], missing_yield=[], category='extra_edge', description='Add 2 wrong edges: N→I and P→H'),
    dict(label='G06: N→I→P chain', inter_edges=[('Nitrogen', 'Irrigation'), ('Irrigation', 'Precipitation')], missing_yield=[], category='chain', description='Add 3-node chain: N→I→P'),
    dict(label='G07: Full chain N→I→P→H', inter_edges=[('Nitrogen', 'Irrigation'), ('Irrigation', 'Precipitation'), ('Precipitation', 'Herbicide')], missing_yield=[], category='chain', description='Full 4-node chain: N→I→P→H (all wrong)'),
    dict(label='G08: Dense (4 wrong edges)', inter_edges=[('Nitrogen', 'Irrigation'), ('Nitrogen', 'Precipitation'), ('Irrigation', 'Precipitation'), ('Irrigation', 'Herbicide')], missing_yield=[], category='extra_edge', description='Dense wrong graph: 4 spurious inter-feature edges'),
    dict(label='G09: No Irrigation→Yield', inter_edges=[], missing_yield=['Irrigation'], category='missing', description='Irrigation disconnected from Yield (feature present, edge absent)'),
    dict(label='G10: No Nitrogen→Yield', inter_edges=[], missing_yield=['Nitrogen'], category='missing', description='Nitrogen disconnected from Yield (feature present, edge absent)'),
]

CAT_COLORS = {'baseline': '#1D9E75', 'extra_edge': '#378ADD', 'chain': '#7F77DD', 'missing': '#D85A30'}


def main():
    print("\n" + "=" * 70)
    print("HIGH-FIDELITY GRAPH SENSITIVITY EXPERIMENT")
    print("=" * 70)
    print(f"  MAX_PERM       : {CONFIG['MAX_PERM']}")
    print(f"  M_RESAMPLES    : {CONFIG['M_RESAMPLES']}")
    print(f"  GLOBAL_K       : {CONFIG['GLOBAL_K']}\n")

    try:
        data = np.load(CONFIG['DATA_PATH'], allow_pickle=True)
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['DATA_PATH']}' not found."); return

    X_train = pd.DataFrame(data['X_train'], columns=FEATURE_NAMES)
    X_test  = pd.DataFrame(data['X_test'],  columns=FEATURE_NAMES)
    y_train = data['y_train'].astype(float)
    y_test  = data['y_test'].astype(float)

    gamma = compute_gamma(X_train)
    print("GT gamma_i (Target Ground Truth Paths):")
    for f, v in gamma.items():
        print(f"  {f:<18}  γ = {v:.4f}")

    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_NAMES)
    Xte = pd.DataFrame(scaler.transform(X_test),      columns=FEATURE_NAMES)

    if HAVE_XGB:
        model = XGBRegressor(
            n_estimators=160, max_depth=5, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9,
            random_state=CONFIG['RANDOM_SEED'], verbosity=0,
            n_jobs=1, tree_method='hist'
        )
    else:
        model = RandomForestRegressor(n_estimators=160, max_depth=6, random_state=CONFIG['RANDOM_SEED'], n_jobs=1)

    model.fit(Xtr.values, y_train)

    all_results = []
    t_total = time.time()

    for i, g in enumerate(GRAPH_VARIANTS):
        print(f"\n[{i+1:02d}/{len(GRAPH_VARIANTS)}] Running {g['label']} ...")
        t0 = time.time()

        dag = build_dag(g['inter_edges'])
        cshap = CausalSHAPRegressor(model=model, dag=dag, gamma=gamma, missing_yield=g['missing_yield'], X_train=Xtr)
        global_abs = cshap.explain_global(Xte, k=CONFIG['GLOBAL_K'], n_perms=CONFIG['MAX_PERM'], M=CONFIG['M_RESAMPLES'])

        m, attributions = compute_metrics(gamma, global_abs)
        elapsed = time.time() - t0

        all_results.append({
            'Label':    g['label'],
            'Category': g['category'],
            **m,
            **attributions,  # Dynamic injection of feature-specific logging columns
            'Time_s':   round(elapsed, 1),
        })

    df = pd.DataFrame(all_results)
    df.to_csv('graph_sensitivity_results.csv', index=False)

    print("\n" + "=" * 85)
    print("FINAL DATASET SUMMARY (Metrics & Logged Feature Attributions)")
    print("=" * 85)
    display_cols = ['Label', 'Pearson_r', 'RMSE', 'Attr_Nitrogen', 'Attr_Irrigation', 'Attr_Precipitation', 'Attr_Herbicide']
    print(df[display_cols].to_string(index=False))

    plot_bar_chart(df, 'graph_sensitivity_bar.png')
    plot_heatmap(df,   'graph_sensitivity_heatmap.png')
    print(f"\nCompleted in { (time.time() - t_total)/60:.1f} minutes.")


if __name__ == '__main__':
    main()


HIGH-FIDELITY GRAPH SENSITIVITY EXPERIMENT
  MAX_PERM       : 500
  M_RESAMPLES    : 128
  GLOBAL_K       : 100

GT gamma_i (Target Ground Truth Paths):
  Nitrogen            γ = 0.3118
  Irrigation          γ = 0.3727
  Precipitation       γ = 0.2457
  Herbicide           γ = 0.0697

[01/11] Running G00: True graph ...
      instance 10/100 ...
      instance 20/100 ...
      instance 30/100 ...
      instance 40/100 ...
      instance 50/100 ...
      instance 60/100 ...
      instance 70/100 ...
      instance 80/100 ...
      instance 90/100 ...
      instance 100/100 ...

[02/11] Running G01: +N→I ...
      instance 10/100 ...
      instance 20/100 ...
      instance 30/100 ...
      instance 40/100 ...
      instance 50/100 ...
      instance 60/100 ...
      instance 70/100 ...
      instance 80/100 ...
      instance 90/100 ...
      instance 100/100 ...

[03/11] Running G02: +I→N ...
      instance 10/100 ...
      instance 20/100 ...
      instance 30/100 ...
      instance 

In [1]:
"""
Weight Sensitivity Experiment — Causal SHAP on Yield Regression
================================================================
Fixes the TRUE causal graph (star DAG, no inter-feature edges) and
perturbs only the gamma (γᵢ) weights — the feature→target importance
priors used inside Causal SHAP.

For yield, the true graph has NO inter-feature edges, so SEM edge weight
perturbations are not applicable (there are no edges to perturb). All
sensitivity comes from gamma misspecification.

ENGINE: deliberately matches the original graph-sensitivity experiment
(graph_sensitivity_yield.py) exactly — same permutation-based Shapley
estimator, same module-level rng, same random instance sampling in
explain_global, same StandardScaler usage — so baseline results are
directly comparable (Pearson ~0.984 on G00).

Gamma variant families
----------------------
  G00 : baseline — true γ from translog elasticities
  G01 : uniform γ (equal weight on all 4 features)
  G02 : top-1 dominant — all weight on Irrigation (highest γ)
  G03 : inverted γ — highest true → lowest assigned, and vice versa
  G04 : scale top feature (Irrigation) ×2, renormalise
  G05 : scale bottom feature (Herbicide) ×5, renormalise
  G06 : zero top feature (Irrigation), renormalise over rest
  G07 : zero bottom feature (Herbicide), renormalise over rest
  G08 : Gaussian noise σ=0.1 added to each γ, clip ≥ 0, renormalise
  G09 : Gaussian noise σ=0.3 added to each γ, clip ≥ 0, renormalise
  G10 : top-2 only (Irrigation + Nitrogen), rest zeroed
  G11 : swap top-2 (Irrigation ↔ Nitrogen), rest unchanged

Outputs
-------
    yield_weight_sensitivity_results.csv
"""

import time
import warnings
from copy import deepcopy
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import networkx as nx
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

#  CONFIG - mirrors original graph sensitivity experiment exactly
CONFIG = {
    'DATA_PATH':   '../datasets/yield_regression_preprocessed.npz',
    'MAX_PERM':    500,
    'M_RESAMPLES': 128,
    'GLOBAL_K':    100,
    'RANDOM_SEED': 42,
}

FEATURE_NAMES = ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']

TRANSLOG_COEFFS = {
    'betaN':  0.35,  'betaI':  0.50,  'betaP':  0.25,  'betaH':  0.05,
    'betaNN': -0.02, 'betaII': -0.03, 'betaPP': -0.01, 'betaHH': -0.005,
    'betaNI':  0.04, 'betaNP':  0.02, 'betaNH':  0.01,
    'betaIP':  0.03, 'betaIH':  0.01, 'betaPH':  0.005,
}

# Module-level rng - matches original script exactly (stateful across all calls)
np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])

# Separate rng for gamma noise variants so noise values are reproducible
# and independent of the Shapley estimation rng
rng_gamma_noise = np.random.default_rng(0)


# Ground-truth gamma
def compute_gamma(X_train: pd.DataFrame) -> Dict[str, float]:
    """
    Derive γᵢ from translog structural equation via elasticities.
    Must be called with RAW (unscaled) X_train — uses log(x).
    """
    for f in FEATURE_NAMES:
        if (X_train[f] <= 0).any():
            raise ValueError(
                f"compute_gamma: non-positive values in '{f}'. "
                "Pass unscaled X_train — StandardScaler output is invalid here."
            )
    c   = TRANSLOG_COEFFS
    lm  = {f: float(np.log(X_train[f]).mean()) for f in FEATURE_NAMES}
    lnN, lnI, lnP, lnH = (lm['Nitrogen'], lm['Irrigation'],
                           lm['Precipitation'], lm['Herbicide'])
    elasticities = {
        'Nitrogen':
            c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI
            + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':
            c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN
            + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation':
            c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN
            + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':
            c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN
            + c['betaIH']*lnI + c['betaPH']*lnP,
    }
    total = sum(elasticities.values())
    return {f: v / total for f, v in elasticities.items()}


#  DAG + SEM - identical to original script
def build_dag(inter_edges: List[Tuple[str, str]]) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for u, v in inter_edges:
        G.add_edge(u, v)
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError(f"Cycle detected: {inter_edges}")
    return G


class LinearMechanisms:
    """Linear-Gaussian SEM fitted on the feature DAG."""

    def __init__(self, dag: nx.DiGraph):
        self.dag  = dag
        self.mech: Dict[str, dict] = {}
        self.topo: List[str] = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        for var in FEATURE_NAMES:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def sample_intervention_batch(
        self, S: Set[str], x_obs: pd.Series, M: int
    ) -> np.ndarray:
        X_samples   = np.empty((M, len(FEATURE_NAMES)))
        feat_to_idx = {f: i for i, f in enumerate(FEATURE_NAMES)}
        simulated   = {}

        for var in self.topo:
            v_idx = feat_to_idx[var]
            if var in S:
                val = float(x_obs[var])
                X_samples[:, v_idx] = val
                simulated[var]      = np.full(M, val)
                continue
            m     = self.mech[var]
            sigma = m['sigma']
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                p_matrix = np.empty((M, len(m['parents'])))
                for pi, p in enumerate(m['parents']):
                    p_matrix[:, pi] = (
                        simulated[p] if p in simulated else float(x_obs[p])
                    )
                mu = m['intercept'] + p_matrix @ m['coef']
            X_samples[:, v_idx] = (
                mu if sigma < 1e-8 else rng.normal(mu, sigma, size=M)
            )
            simulated[var] = X_samples[:, v_idx]

        return X_samples


#  CAUSAL SHAP ENGINE - identical to original script
class CausalSHAPRegressor:
    """
    Permutation-based Causal SHAP estimator.
    Matches the original graph_sensitivity_yield.py engine exactly.
    """

    def __init__(
        self,
        model,
        dag:           nx.DiGraph,
        gamma:         Dict[str, float],
        missing_yield: List[str],
        X_train:       pd.DataFrame,
    ):
        self.model    = model
        self.features = list(X_train.columns)

        # Zero-out γ for features declared as disconnected from Yield,
        # then renormalise - matches original missing_yield logic
        eff = {f: (0.0 if f in missing_yield else gamma[f]) for f in self.features}
        tot = sum(eff.values())
        self.gamma = {f: v / tot for f, v in eff.items()} if tot > 1e-12 else eff

        self.sem  = LinearMechanisms(dag).fit(X_train)
        self._Ef  = float(model.predict(X_train.values).mean())

    def _vc_batch(self, S: Set[str], x: pd.Series, M: int) -> float:
        X_mat = self.sem.sample_intervention_batch(S, x, M)
        return float(np.mean(self.model.predict(X_mat)))

    def explain_instance(self, x: pd.Series, n_perms: int, M: int) -> pd.Series:
        fx  = float(self.model.predict(x.values.reshape(1, -1))[0])
        phi = np.zeros(len(self.features), dtype=float)

        for _ in range(n_perms):
            perm   = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc_batch(S, x, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc_batch(S_next, x, M)
                phi[self.features.index(feat)] += self.gamma[feat] * (v_curr - v_prev)
                v_prev, S = v_curr, S_next
        phi /= n_perms

        s = float(np.sum(phi))
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)

        return pd.Series(phi, index=self.features)

    def explain_global(
        self, X_test: pd.DataFrame, k: int, n_perms: int, M: int
    ) -> pd.Series:
        k   = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)   # random sample, matches original
        phi_list = []
        for t, i in enumerate(idx, 1):
            if t % 10 == 0:
                print(f"      instance {t}/{k} ...", flush=True)
            phi_list.append(self.explain_instance(X_test.iloc[i], n_perms, M))
        return pd.concat(phi_list, axis=1).abs().mean(axis=1)


# Metrics
def compute_metrics(
    true_gamma: Dict[str, float], global_abs: pd.Series
) -> Tuple[dict, dict]:
    gt_vec   = np.array([true_gamma[f]        for f in FEATURE_NAMES])
    raw      = np.array([global_abs.get(f, 0) for f in FEATURE_NAMES])
    shap_vec = raw / raw.sum() if raw.sum() > 1e-12 else raw

    pr, _ = pearsonr(shap_vec,  gt_vec)
    sr, _ = spearmanr(shap_vec, gt_vec)
    cos   = float(np.dot(shap_vec, gt_vec) /
                  (np.linalg.norm(shap_vec) * np.linalg.norm(gt_vec) + 1e-12))
    rmse  = float(np.sqrt(np.mean((shap_vec - gt_vec) ** 2)))
    mae   = float(np.mean(np.abs(shap_vec - gt_vec)))

    metrics = {
        'Pearson_r':  round(float(pr),  4),
        'Spearman_r': round(float(sr),  4),
        'Cosine':     round(cos,        4),
        'RMSE':       round(rmse,       4),
        'MAE':        round(mae,        4),
    }
    attrs = {f"Attr_{f}": round(shap_vec[i], 4) for i, f in enumerate(FEATURE_NAMES)}
    return metrics, attrs


# Gamma variant builder
def build_gamma_variants(true_gamma: Dict[str, float]) -> List[dict]:
    """
    All returned gamma dicts are normalised to sum to 1.
    Noise is drawn from rng_gamma_noise (fixed seed=0) so variant
    values are reproducible regardless of how many Shapley samples
    the main rng has consumed.
    """
    def norm(g: Dict[str, float]) -> Dict[str, float]:
        g   = {f: max(v, 0.0) for f, v in g.items()}
        tot = sum(g.values())
        if tot < 1e-12:
            return {f: 1.0 / len(FEATURE_NAMES) for f in FEATURE_NAMES}
        return {f: v / tot for f, v in g.items()}

    tg       = true_gamma
    by_val   = sorted(FEATURE_NAMES, key=lambda f: tg[f], reverse=True)
    top_f    = by_val[0]    # Irrigation (γ≈0.373)
    bot_f    = by_val[-1]   # Herbicide  (γ≈0.070)
    top2     = by_val[:2]   # Irrigation, Nitrogen
    n        = len(FEATURE_NAMES)

    variants = [
        # G00 - true baseline
        dict(label='G00: true gamma (baseline)',
             gamma=dict(tg)),

        # G01 - uniform
        dict(label='G01: uniform gamma',
             gamma=norm({f: 1.0 / n for f in FEATURE_NAMES})),

        # G02 - top-1 dominant (all weight on Irrigation)
        dict(label=f'G02: top-1 dominant ({top_f})',
             gamma=norm({f: (1.0 if f == top_f else 0.0) for f in FEATURE_NAMES})),

        # G03 - inverted rank
        dict(label='G03: inverted gamma',
             gamma=norm({f: tg[by_val[n - 1 - i]] for i, f in enumerate(by_val)})),

        # G04 - scale top feature ×2
        dict(label=f'G04: top feature ({top_f}) ×2',
             gamma=norm({f: (tg[f] * 2 if f == top_f else tg[f]) for f in FEATURE_NAMES})),

        # G05 - scale bottom feature ×5
        dict(label=f'G05: bottom feature ({bot_f}) ×5',
             gamma=norm({f: (tg[f] * 5 if f == bot_f else tg[f]) for f in FEATURE_NAMES})),

        # G06 - zero top feature
        dict(label=f'G06: zero top feature ({top_f})',
             gamma=norm({f: (0.0 if f == top_f else tg[f]) for f in FEATURE_NAMES})),

        # G07 - zero bottom feature
        dict(label=f'G07: zero bottom feature ({bot_f})',
             gamma=norm({f: (0.0 if f == bot_f else tg[f]) for f in FEATURE_NAMES})),

        # G08 - Gaussian noise σ=0.1
        dict(label='G08: gamma + noise σ=0.1',
             gamma=norm({f: tg[f] + float(rng_gamma_noise.normal(0, 0.1))
                         for f in FEATURE_NAMES})),

        # G09 - Gaussian noise σ=0.3
        dict(label='G09: gamma + noise σ=0.3',
             gamma=norm({f: tg[f] + float(rng_gamma_noise.normal(0, 0.3))
                         for f in FEATURE_NAMES})),

        # G10 - top-2 features only (Irrigation + Nitrogen), rest zeroed
        dict(label=f'G10: top-2 only ({top2[0]}, {top2[1]})',
             gamma=norm({f: (tg[f] if f in top2 else 0.0) for f in FEATURE_NAMES})),

        # G11 - swap top-2 (Irrigation ↔ Nitrogen), rest unchanged
        dict(label=f'G11: swap top-2 ({top2[0]} ↔ {top2[1]})',
             gamma=norm({
                 f: (tg[top2[1]] if f == top2[0] else
                     tg[top2[0]] if f == top2[1] else
                     tg[f])
                 for f in FEATURE_NAMES
             })),
    ]

    # Print the variant table for inspection
    print(f"\n  {'Label':<45}  {'N':>7}  {'I':>7}  {'P':>7}  {'H':>7}")
    print(f"  {'-'*45}  {'-'*7}  {'-'*7}  {'-'*7}  {'-'*7}")
    for v in variants:
        g = v['gamma']
        print(f"  {v['label']:<45}  "
              f"{g['Nitrogen']:>7.4f}  {g['Irrigation']:>7.4f}  "
              f"{g['Precipitation']:>7.4f}  {g['Herbicide']:>7.4f}")

    return variants


# Main
def main():
    print("\n" + "=" * 75)
    print("YIELD WEIGHT SENSITIVITY — GAMMA PERTURBATION EXPERIMENT")
    print("=" * 75)
    print(f"  Engine        : permutation-based (MAX_PERM={CONFIG['MAX_PERM']})")
    print(f"  MC samples    : {CONFIG['M_RESAMPLES']}")
    print(f"  Test instances: {CONFIG['GLOBAL_K']}")
    print(f"  True graph    : star DAG (no inter-feature edges)")

    # ── Load data ────────────────────────────────────────────────
    data        = np.load(CONFIG['DATA_PATH'], allow_pickle=True)
    X_train_raw = pd.DataFrame(data['X_train'], columns=FEATURE_NAMES)
    X_test_raw  = pd.DataFrame(data['X_test'],  columns=FEATURE_NAMES)
    y_train     = data['y_train'].astype(float)

    # Gamma from RAW data (translog needs log(x) > 0)
    true_gamma = compute_gamma(X_train_raw)

    print(f"\n  True gamma:")
    for f, v in true_gamma.items():
        print(f"    {f:<18}  γ = {v:.4f}")

    # Scale AFTER computing gamma - matches original script
    scaler = StandardScaler()
    Xtr    = pd.DataFrame(scaler.fit_transform(X_train_raw),
                          columns=FEATURE_NAMES)
    Xte    = pd.DataFrame(scaler.transform(X_test_raw),
                          columns=FEATURE_NAMES)

    # ── Train model (once, shared across all variants) ────────────
    model = XGBRegressor(
        n_estimators=160, max_depth=5, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9,
        random_state=CONFIG['RANDOM_SEED'],
        verbosity=0, n_jobs=1, tree_method='hist',
    )
    model.fit(Xtr.values, y_train)

    # True graph - star DAG, no inter-feature edges
    true_dag = build_dag([])

    # ── Build gamma variants ──────────────────────────────────────
    variants = build_gamma_variants(true_gamma)
    print(f"\n  Total gamma variants: {len(variants)}\n")

    all_results = []
    t_total     = time.time()

    for v_idx, v in enumerate(variants, 1):
        label = v['label']
        gamma = v['gamma']
        print(f"\n[{v_idx:02d}/{len(variants)}] {label}")

        t0    = time.time()
        cshap = CausalSHAPRegressor(
            model=model,
            dag=true_dag,
            gamma=gamma,
            missing_yield=[],   # no missing edges - pure gamma perturbation
            X_train=Xtr,
        )
        global_abs = cshap.explain_global(
            Xte,
            k=CONFIG['GLOBAL_K'],
            n_perms=CONFIG['MAX_PERM'],
            M=CONFIG['M_RESAMPLES'],
        )
        elapsed = time.time() - t0

        m, attrs = compute_metrics(true_gamma, global_abs)
        print(
            f"  Pearson={m['Pearson_r']:.4f} | Spearman={m['Spearman_r']:.4f} | "
            f"Cosine={m['Cosine']:.4f} | RMSE={m['RMSE']:.4f} | "
            f"MAE={m['MAE']:.4f} | {elapsed:.1f}s"
        )

        all_results.append({
            'Label':      label,
            'Gamma_N':    round(gamma['Nitrogen'],     4),
            'Gamma_I':    round(gamma['Irrigation'],   4),
            'Gamma_P':    round(gamma['Precipitation'],4),
            'Gamma_H':    round(gamma['Herbicide'],    4),
            **m,
            **attrs,
            'Time_s':     round(elapsed, 1),
        })

    df = pd.DataFrame(all_results)
    out = 'yield_weight_sensitivity_results.csv'
    df.to_csv(out, index=False)

    print("\n" + "=" * 105)
    print("RESULTS SUMMARY — YIELD GAMMA SENSITIVITY")
    print("=" * 105)
    disp = ['Label', 'Gamma_N', 'Gamma_I', 'Gamma_P', 'Gamma_H',
            'Pearson_r', 'Spearman_r', 'Cosine', 'RMSE', 'MAE']
    print(df[disp].to_string(index=False))
    print(f"\nTotal runtime: {(time.time() - t_total) / 60:.1f} min")
    print(f"Saved → {out}\n")


if __name__ == '__main__':
    main()


YIELD WEIGHT SENSITIVITY — GAMMA PERTURBATION EXPERIMENT
  Engine        : permutation-based (MAX_PERM=500)
  MC samples    : 128
  Test instances: 100
  True graph    : star DAG (no inter-feature edges)

  True gamma:
    Nitrogen            γ = 0.3118
    Irrigation          γ = 0.3727
    Precipitation       γ = 0.2457
    Herbicide           γ = 0.0697

  Label                                                N        I        P        H
  ---------------------------------------------  -------  -------  -------  -------
  G00: true gamma (baseline)                      0.3118   0.3727   0.2457   0.0697
  G01: uniform gamma                              0.2500   0.2500   0.2500   0.2500
  G02: top-1 dominant (Irrigation)                0.0000   1.0000   0.0000   0.0000
  G03: inverted gamma                             0.2457   0.0697   0.3118   0.3727
  G04: top feature (Irrigation) ×2                0.2272   0.5431   0.1790   0.0508
  G05: bottom feature (Herbicide) ×5              0

In [1]:
"""
Gamma Inversion Diagnostic — Causal SHAP
=========================================
Investigates WHY inverted gamma produces mild degradation for yield
(Pearson=0.83) but catastrophic collapse for soil (Pearson=-0.39).

Core question: is the inversion being "cancelled" by the model's own
marginal contribution signal? And if so, how much of the final
attribution comes from gamma vs the model?

Experiments run:
  (A) Decomposition: separate gamma contribution from model contribution
      - Run with true gamma   → attribution_true
      - Run with uniform gamma → attribution_uniform  (gamma flattened out)
      - Run with inverted gamma → attribution_inverted
      - Compare: how much does attribution shift relative to gamma shift?

  (B) Gamma sweep: gradually blend from true → inverted gamma
      alpha=0.0 → true gamma
      alpha=0.5 → midpoint
      alpha=1.0 → fully inverted
      Tracks Pearson r at each step to show the degradation curve.

  (C) Model marginal baseline: estimate avg marginal contribution per
      feature WITHOUT any gamma weighting (gamma=uniform), which isolates
      the pure model signal. Then compare to gamma-weighted attributions.

  (D) Counterfactual: what gamma inversion WOULD look like if the model
      had no opinion (uniform marginals). Shows the theoretical floor.

Outputs:
    gamma_inversion_diagnostic_yield.csv
    gamma_inversion_diagnostic_soil.csv
    (printed decomposition tables for both datasets)
"""

import time
import itertools
import math
import warnings
from copy import deepcopy
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import networkx as nx
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# Config
CONFIG = {
    'SOIL_DATA_PATH':         '../datasets/data_complete.csv',
    'SOIL_GROUND_TRUTH_PATH': '../datasets/ground_truth.csv',
    'YIELD_DATA_PATH':        '../datasets/yield_regression_preprocessed.npz',
    'TEST_SIZE':     0.20,
    'RANDOM_SEED':   42,

    # Yield uses permutation engine (matches original experiment)
    'YIELD_MAX_PERM':  500,
    'YIELD_M':         128,
    'YIELD_K':         100,

    # Soil uses exact engine (matches G00-G08 experiment)
    'SOIL_M':          128,
    'SOIL_K':          100,

    # Gamma sweep: number of alpha steps from true->inverted
    'SWEEP_STEPS': 11,
}

SOIL_FEATURES   = ['Organic Carbon','Nitrogen','Phosphorus','Potassium','pH','Zinc','Electrical Conductivity']
SOIL_COL_MAP    = {'OC':'Organic Carbon','N':'Nitrogen','P':'Phosphorus','K':'Potassium','PH':'pH','ZN':'Zinc','EC':'Electrical Conductivity'}
SOIL_TRUE_EDGES = [('Organic Carbon','pH'),('pH','Zinc')]

YIELD_FEATURES  = ['Nitrogen','Irrigation','Precipitation','Herbicide']
YIELD_TRANSLOG  = {
    'betaN':0.35,'betaI':0.50,'betaP':0.25,'betaH':0.05,
    'betaNN':-0.02,'betaII':-0.03,'betaPP':-0.01,'betaHH':-0.005,
    'betaNI':0.04,'betaNP':0.02,'betaNH':0.01,
    'betaIP':0.03,'betaIH':0.01,'betaPH':0.005,
}

# Module-level rng - yield engine uses this (stateful, matches original)
np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])


# Gamma functions
def compute_soil_gamma(path: str) -> Dict[str, float]:
    gt  = pd.read_csv(path)
    net = pd.DataFrame({
        'Organic Carbon':          gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                gt['N_base']   - gt['N_penalty'],
        'Phosphorus':              gt['P_base']   - gt['P_penalty'],
        'Potassium':               gt['K_base']   - gt['K_penalty'],
        'pH':                     -gt['PH_penalty'],
        'Zinc':                    gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    m = net.abs().mean()
    return (m / m.sum()).to_dict()


def compute_yield_gamma(X_raw: pd.DataFrame) -> Dict[str, float]:
    for f in YIELD_FEATURES:
        if (X_raw[f] <= 0).any():
            raise ValueError(f"compute_yield_gamma: non-positive values in '{f}'. Pass unscaled data.")
    c   = YIELD_TRANSLOG
    lm  = {f: float(np.log(X_raw[f]).mean()) for f in YIELD_FEATURES}
    lnN, lnI, lnP, lnH = lm['Nitrogen'], lm['Irrigation'], lm['Precipitation'], lm['Herbicide']
    e = {
        'Nitrogen':      c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':    c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation': c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':     c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN + c['betaIH']*lnI + c['betaPH']*lnP,
    }
    tot = sum(e.values())
    return {f: v/tot for f, v in e.items()}


def normalise(g: Dict[str, float], features: List[str]) -> Dict[str, float]:
    g   = {f: max(v, 0.0) for f, v in g.items()}
    tot = sum(g.values())
    if tot < 1e-12:
        return {f: 1.0/len(features) for f in features}
    return {f: v/tot for f, v in g.items()}


def invert_gamma(true_gamma: Dict[str, float], features: List[str]) -> Dict[str, float]:
    """Rank-inversion: highest γ → lowest value assigned, and vice versa."""
    by_val = sorted(features, key=lambda f: true_gamma[f], reverse=True)
    n      = len(features)
    return normalise({f: true_gamma[by_val[n-1-i]] for i, f in enumerate(by_val)}, features)


def blend_gamma(true_gamma: Dict[str, float],
                inv_gamma:  Dict[str, float],
                alpha:      float,
                features:   List[str]) -> Dict[str, float]:
    """Linear interpolation: alpha=0 → true, alpha=1 → inverted."""
    blended = {f: (1-alpha)*true_gamma[f] + alpha*inv_gamma[f] for f in features}
    return normalise(blended, features)


def uniform_gamma(features: List[str]) -> Dict[str, float]:
    return {f: 1.0/len(features) for f in features}


# Dag + sem
def build_dag(features: List[str], edges: List[Tuple[str,str]]) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(features)
    for u,v in edges:
        G.add_edge(u, v)
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError("Cycle detected")
    return G


class LinearMechanisms:
    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag      = dag
        self.features = features
        self.mech:    Dict[str, dict] = {}
        self.topo:    List[str] = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {'parents':[],'intercept':float(X[var].mean()),'coef':None,'sigma':float(X[var].std(ddof=0))}
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {'parents':parents,'intercept':float(lr.intercept_),'coef':lr.coef_.astype(float),'sigma':float(np.std(resid,ddof=0))}
        return self

    def sample_batch(self, S: Set[str], x_obs: pd.Series, M: int) -> np.ndarray:
        X_out       = np.empty((M, len(self.features)))
        feat_to_idx = {f: i for i, f in enumerate(self.features)}
        simulated   = {}
        for var in self.topo:
            v_idx = feat_to_idx[var]
            if var in S:
                val = float(x_obs[var])
                X_out[:, v_idx] = val
                simulated[var]  = np.full(M, val)
                continue
            m = self.mech[var]
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                pm = np.empty((M, len(m['parents'])))
                for pi, p in enumerate(m['parents']):
                    pm[:, pi] = simulated[p] if p in simulated else float(x_obs[p])
                mu = m['intercept'] + pm @ m['coef']
            X_out[:, v_idx] = mu if m['sigma'] < 1e-8 else rng.normal(mu, m['sigma'], size=M)
            simulated[var]  = X_out[:, v_idx]
        return X_out


# Engines
class PermutationEngine:
    """Yield engine — matches original graph_sensitivity_yield.py exactly."""
    def __init__(self, model, sem: LinearMechanisms, gamma: Dict[str,float], X_train: pd.DataFrame):
        self.model    = model
        self.features = list(X_train.columns)
        self.gamma    = gamma
        self.sem      = sem
        self._Ef      = float(model.predict(X_train.values).mean())

    def _vc(self, S: Set[str], x: pd.Series, M: int) -> float:
        return float(np.mean(self.model.predict(self.sem.sample_batch(S, x, M))))

    def explain_instance(self, x: pd.Series, n_perms: int, M: int) -> pd.Series:
        fx  = float(self.model.predict(x.values.reshape(1,-1))[0])
        phi = np.zeros(len(self.features))
        for _ in range(n_perms):
            perm   = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc(S, x, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc(S_next, x, M)
                phi[self.features.index(feat)] += self.gamma[feat] * (v_curr - v_prev)
                v_prev, S = v_curr, S_next
        phi /= n_perms
        s = float(phi.sum())
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame, k: int, n_perms: int, M: int) -> pd.Series:
        k   = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        phi_list = []
        for t, i in enumerate(idx, 1):
            if t % 10 == 0:
                print(f"        instance {t}/{k} ...", flush=True)
            phi_list.append(self.explain_instance(X_test.iloc[i], n_perms, M))
        return pd.concat(phi_list, axis=1).abs().mean(axis=1)


class ExactEngine:
    """Soil engine — exact 2^n subset enumeration."""
    def __init__(self, model, sem: LinearMechanisms, gamma: Dict[str,float], X_train: pd.DataFrame):
        self.model    = model
        self.features = list(X_train.columns)
        self.n        = len(self.features)
        self.gamma    = gamma
        self.sem      = sem
        self._Ef      = float(model.predict(X_train.values).mean())
        self._kw      = {}
        for sz in range(self.n):
            self._kw[sz] = float(math.factorial(sz) * math.factorial(self.n-sz-1) / math.factorial(self.n))

    def _v(self, S: frozenset, x: pd.Series, M: int) -> float:
        return float(np.mean(self.model.predict(self.sem.sample_batch(S, x, M))))

    def explain_instance(self, x: pd.Series, M: int) -> pd.Series:
        fx    = float(self.model.predict(x.values.reshape(1,-1))[0])
        phi   = np.zeros(self.n)
        cache = {}
        for r in range(self.n+1):
            for sub in itertools.combinations(self.features, r):
                S = frozenset(sub)
                cache[S] = self._v(S, x, M)
        for idx, feat in enumerate(self.features):
            rest = [f for f in self.features if f != feat]
            for r in range(self.n):
                for sub in itertools.combinations(rest, r):
                    S = frozenset(sub)
                    phi[idx] += self._kw[len(sub)] * self.gamma[feat] * (cache[frozenset(set(sub)|{feat})] - cache[S])
        s = float(phi.sum())
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame, k: int, M: int) -> pd.Series:
        k = min(k, len(X_test))
        phi_list = []
        for i in range(k):
            if (i+1) % 20 == 0:
                print(f"        instance {i+1}/{k} ...", flush=True)
            phi_list.append(self.explain_instance(X_test.iloc[i], M))
        return pd.concat(phi_list, axis=1).abs().mean(axis=1)


# Metrics
def metrics(true_gamma: Dict[str,float], global_abs: pd.Series, features: List[str]) -> dict:
    gt   = np.array([true_gamma[f]        for f in features])
    raw  = np.array([global_abs.get(f,0) for f in features])
    sv   = raw / raw.sum() if raw.sum() > 1e-12 else raw
    pr,_ = pearsonr(sv, gt)
    sr,_ = spearmanr(sv, gt)
    cos  = float(np.dot(sv, gt) / (np.linalg.norm(sv) * np.linalg.norm(gt) + 1e-12))
    rmse = float(np.sqrt(np.mean((sv - gt)**2)))
    return {
        'Pearson_r':  round(float(pr), 4),
        'Spearman_r': round(float(sr), 4),
        'Cosine':     round(cos,       4),
        'RMSE':       round(rmse,      4),
        'shap_vec':   sv,
    }


# Decomposition analysis
def decomposition_analysis(true_gamma: Dict[str,float],
                           inv_gamma:  Dict[str,float],
                           global_true: pd.Series,
                           global_inv:  pd.Series,
                           global_unif: pd.Series,
                           features: List[str]) -> None:
    """
    Prints a table showing how much of the attribution shift is explained
    by gamma vs the underlying model marginal signal.
    """
    by_val = sorted(features, key=lambda f: true_gamma[f], reverse=True)

    gt   = np.array([true_gamma[f]  for f in features])
    inv  = np.array([inv_gamma[f]   for f in features])
    unif = np.array([1/len(features) for f in features])

    def norm(v): return v / v.sum() if v.sum() > 1e-12 else v
    sv_true = norm(np.array([global_true.get(f,0) for f in features]))
    sv_inv  = norm(np.array([global_inv.get(f,0)  for f in features]))
    sv_unif = norm(np.array([global_unif.get(f,0) for f in features]))

    # Model marginal signal = attribution with uniform gamma (gamma cancels out)
    model_signal = sv_unif  # with uniform gamma, phi ∝ avg_marginal

    print(f"\n  {'Feature':<28} {'γ_true':>7} {'γ_inv':>7} {'model_signal':>13} {'attr_true':>10} {'attr_inv':>9}")
    print(f"  {'-'*28} {'-'*7} {'-'*7} {'-'*13} {'-'*10} {'-'*9}")
    for i, f in enumerate(by_val):
        fi = features.index(f)
        print(f"  {f:<28} {gt[fi]:>7.4f} {inv[fi]:>7.4f} {model_signal[fi]:>13.4f} {sv_true[fi]:>10.4f} {sv_inv[fi]:>9.4f}")

    # Key ratio: how much did attribution shift vs how much did gamma shift?
    gamma_shift = np.abs(inv - gt)
    attr_shift  = np.abs(sv_inv - sv_true)
    print(f"\n  Gamma shift (L1):       {gamma_shift.sum():.4f}")
    print(f"  Attribution shift (L1): {attr_shift.sum():.4f}")
    print(f"  Shift ratio (attr/gamma): {attr_shift.sum()/gamma_shift.sum():.3f}")
    print(f"  (ratio < 1 means model partially absorbs the inversion)")

    # Theoretical prediction: if attribution = gamma × model_signal (normalised)
    theoretical = norm(inv * model_signal)
    pr_th, _ = pearsonr(theoretical, gt)
    print(f"\n  Theoretical Pearson (γ_inv × model_signal): {pr_th:.4f}")
    pr_actual, _ = pearsonr(sv_inv, gt)
    print(f"  Actual Pearson (G03 inverted):               {pr_actual:.4f}")


# Gamma sweep
def run_gamma_sweep(true_gamma, inv_gamma, features, model, sem, X_train, X_test,
                    engine_type, cfg) -> List[dict]:
    """
    Sweeps alpha from 0 (true gamma) to 1 (inverted gamma) in SWEEP_STEPS steps.
    Returns list of {alpha, gamma_vec, Pearson_r, Spearman_r, Cosine, RMSE}.
    """
    alphas  = np.linspace(0, 1, CONFIG['SWEEP_STEPS'])
    results = []
    for alpha in alphas:
        g = blend_gamma(true_gamma, inv_gamma, float(alpha), features)
        print(f"    alpha={alpha:.1f}  γ={[round(g[f],3) for f in features]}", flush=True)

        if engine_type == 'permutation':
            engine = PermutationEngine(model, sem, g, X_train)
            g_abs  = engine.explain_global(X_test, cfg['YIELD_K'], cfg['YIELD_MAX_PERM'], cfg['YIELD_M'])
        else:
            engine = ExactEngine(model, sem, g, X_train)
            g_abs  = engine.explain_global(X_test, cfg['SOIL_K'], cfg['SOIL_M'])

        m = metrics(true_gamma, g_abs, features)
        results.append({
            'alpha':      round(float(alpha), 2),
            'Pearson_r':  m['Pearson_r'],
            'Spearman_r': m['Spearman_r'],
            'Cosine':     m['Cosine'],
            'RMSE':       m['RMSE'],
            **{f"gamma_{f[:3]}": round(g[f], 4) for f in features},
        })
        print(f"      Pearson={m['Pearson_r']:.4f} | Spearman={m['Spearman_r']:.4f} | "
              f"Cosine={m['Cosine']:.4f} | RMSE={m['RMSE']:.4f}")
    return results


# Main
def main():
    print("\n" + "="*80)
    print("GAMMA INVERSION DIAGNOSTIC — YIELD + SOIL")
    print("="*80)

    # Yield
    print("\n" + "="*80)
    print("YIELD DATASET")
    print("="*80)

    data        = np.load(CONFIG['YIELD_DATA_PATH'], allow_pickle=True)
    X_raw_train = pd.DataFrame(data['X_train'], columns=YIELD_FEATURES)
    X_raw_test  = pd.DataFrame(data['X_test'],  columns=YIELD_FEATURES)
    y_train_y   = data['y_train'].astype(float)

    true_gamma_y = compute_yield_gamma(X_raw_train)
    inv_gamma_y  = invert_gamma(true_gamma_y, YIELD_FEATURES)
    unif_gamma_y = uniform_gamma(YIELD_FEATURES)

    print(f"\n  True gamma: {true_gamma_y}")
    print(f"  Inverted:   {inv_gamma_y}")

    scaler_y = StandardScaler()
    Xtr_y = pd.DataFrame(scaler_y.fit_transform(X_raw_train), columns=YIELD_FEATURES)
    Xte_y = pd.DataFrame(scaler_y.transform(X_raw_test),      columns=YIELD_FEATURES)

    model_y = XGBRegressor(n_estimators=160, max_depth=5, learning_rate=0.08,
                           subsample=0.9, colsample_bytree=0.9,
                           random_state=CONFIG['RANDOM_SEED'], verbosity=0, n_jobs=1, tree_method='hist')
    model_y.fit(Xtr_y.values, y_train_y)

    dag_y = build_dag(YIELD_FEATURES, [])
    sem_y = LinearMechanisms(dag_y, YIELD_FEATURES).fit(Xtr_y)

    print("\n  [A] Running true / inverted / uniform gamma...")
    for label, g in [("true", true_gamma_y), ("inverted", inv_gamma_y), ("uniform", unif_gamma_y)]:
        print(f"\n  Running {label} gamma...", flush=True)
        engine = PermutationEngine(model_y, sem_y, g, Xtr_y)
        g_abs  = engine.explain_global(Xte_y, CONFIG['YIELD_K'], CONFIG['YIELD_MAX_PERM'], CONFIG['YIELD_M'])
        m      = metrics(true_gamma_y, g_abs, YIELD_FEATURES)
        if label == "true":    gabt_y = g_abs
        elif label=="inverted": gabi_y = g_abs
        else:                   gabu_y = g_abs
        print(f"    Pearson={m['Pearson_r']:.4f} | Spearman={m['Spearman_r']:.4f} | "
              f"Cosine={m['Cosine']:.4f} | RMSE={m['RMSE']:.4f}")

    print("\n  [B] Decomposition analysis:")
    decomposition_analysis(true_gamma_y, inv_gamma_y, gabt_y, gabi_y, gabu_y, YIELD_FEATURES)

    print("\n  [C] Gamma sweep (true → inverted in 11 steps):")
    sweep_y = run_gamma_sweep(true_gamma_y, inv_gamma_y, YIELD_FEATURES,
                               model_y, sem_y, Xtr_y, Xte_y, 'permutation', CONFIG)

    df_y = pd.DataFrame(sweep_y)
    df_y.to_csv('gamma_inversion_diagnostic_yield.csv', index=False)
    print(f"\n  Yield sweep results:")
    print(df_y[['alpha','Pearson_r','Spearman_r','Cosine','RMSE']].to_string(index=False))

    # Soil
    print("\n\n" + "="*80)
    print("SOIL DATASET")
    print("="*80)

    dc       = pd.read_csv(CONFIG['SOIL_DATA_PATH']).rename(columns=SOIL_COL_MAP)
    Xtr_s_raw, Xte_s_raw, ytr_s, _ = train_test_split(
        dc[SOIL_FEATURES], dc['SPI_CONTINUOUS'],
        test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True)

    true_gamma_s = compute_soil_gamma(CONFIG['SOIL_GROUND_TRUTH_PATH'])
    inv_gamma_s  = invert_gamma(true_gamma_s, SOIL_FEATURES)
    unif_gamma_s = uniform_gamma(SOIL_FEATURES)

    print(f"\n  True gamma: {true_gamma_s}")
    print(f"  Inverted:   {inv_gamma_s}")

    scaler_s = StandardScaler()
    Xtr_s = pd.DataFrame(scaler_s.fit_transform(Xtr_s_raw), columns=SOIL_FEATURES, index=Xtr_s_raw.index)
    Xte_s = pd.DataFrame(scaler_s.transform(Xte_s_raw),     columns=SOIL_FEATURES, index=Xte_s_raw.index)

    model_s = XGBRegressor(n_estimators=160, max_depth=5, learning_rate=0.08,
                           subsample=0.9, colsample_bytree=0.9,
                           random_state=CONFIG['RANDOM_SEED'], verbosity=0, n_jobs=1, tree_method='hist')
    model_s.fit(Xtr_s.values, ytr_s.values)

    dag_s = build_dag(SOIL_FEATURES, SOIL_TRUE_EDGES)
    sem_s = LinearMechanisms(dag_s, SOIL_FEATURES).fit(Xtr_s)

    print("\n  [A] Running true / inverted / uniform gamma...")
    for label, g in [("true", true_gamma_s), ("inverted", inv_gamma_s), ("uniform", unif_gamma_s)]:
        print(f"\n  Running {label} gamma...", flush=True)
        engine = ExactEngine(model_s, sem_s, g, Xtr_s)
        g_abs  = engine.explain_global(Xte_s, CONFIG['SOIL_K'], CONFIG['SOIL_M'])
        m      = metrics(true_gamma_s, g_abs, SOIL_FEATURES)
        if label == "true":    gabt_s = g_abs
        elif label=="inverted": gabi_s = g_abs
        else:                   gabu_s = g_abs
        print(f"    Pearson={m['Pearson_r']:.4f} | Spearman={m['Spearman_r']:.4f} | "
              f"Cosine={m['Cosine']:.4f} | RMSE={m['RMSE']:.4f}")

    print("\n  [B] Decomposition analysis:")
    decomposition_analysis(true_gamma_s, inv_gamma_s, gabt_s, gabi_s, gabu_s, SOIL_FEATURES)

    print("\n  [C] Gamma sweep (true → inverted in 11 steps):")
    sweep_s = run_gamma_sweep(true_gamma_s, inv_gamma_s, SOIL_FEATURES,
                               model_s, sem_s, Xtr_s, Xte_s, 'exact', CONFIG)

    df_s = pd.DataFrame(sweep_s)
    df_s.to_csv('gamma_inversion_diagnostic_soil.csv', index=False)
    print(f"\n  Soil sweep results:")
    print(df_s[['alpha','Pearson_r','Spearman_r','Cosine','RMSE']].to_string(index=False))

    # Cross-dataset comparison
    print("\n\n" + "="*80)
    print("CROSS-DATASET COMPARISON — Why yield stays high, soil collapses")
    print("="*80)

    print(f"\n  Gamma concentration (Gini coefficient):")
    def gini(g, features):
        v = sorted([g[f] for f in features])
        n = len(v)
        return sum((2*i - n - 1)*v[i] for i in range(n)) / (n * sum(v))

    print(f"    Yield gamma Gini = {gini(true_gamma_y, YIELD_FEATURES):.4f}  (more balanced → inversion less harmful)")
    print(f"    Soil gamma Gini  = {gini(true_gamma_s, SOIL_FEATURES):.4f}  (more concentrated → inversion catastrophic)")

    print(f"\n  Top feature dominance ratio (γ_max / γ_min):")
    def dom_ratio(g, features):
        vals = [g[f] for f in features]
        return max(vals) / min(vals)

    print(f"    Yield: {dom_ratio(true_gamma_y, YIELD_FEATURES):.1f}×  (Irrigation/Herbicide)")
    print(f"    Soil:  {dom_ratio(true_gamma_s, SOIL_FEATURES):.1f}×  (Nitrogen/Zinc)")

    print(f"\n  Shift ratio (how much attribution shifts per unit gamma shift):")
    def shift_ratio(true_g, inv_g, g_abs_true, g_abs_inv, features):
        gt  = np.array([true_g[f] for f in features])
        inv = np.array([inv_g[f]  for f in features])
        def norm(v): return v/v.sum() if v.sum()>1e-12 else v
        sv_t = norm(np.array([g_abs_true.get(f,0) for f in features]))
        sv_i = norm(np.array([g_abs_inv.get(f,0)  for f in features]))
        return np.abs(sv_i-sv_t).sum() / np.abs(inv-gt).sum()

    print(f"    Yield: {shift_ratio(true_gamma_y, inv_gamma_y, gabt_y, gabi_y, YIELD_FEATURES):.3f}")
    print(f"    Soil:  {shift_ratio(true_gamma_s, inv_gamma_s, gabt_s, gabi_s, SOIL_FEATURES):.3f}")

    print(f"\nAll done. CSVs: gamma_inversion_diagnostic_yield.csv, gamma_inversion_diagnostic_soil.csv\n")


if __name__ == '__main__':
    main()


GAMMA INVERSION DIAGNOSTIC — YIELD + SOIL

YIELD DATASET

  True gamma: {'Nitrogen': 0.31182036432818405, 'Irrigation': 0.3727391817981272, 'Precipitation': 0.24573773812228175, 'Herbicide': 0.069702715751407}
  Inverted:   {'Irrigation': 0.069702715751407, 'Nitrogen': 0.24573773812228175, 'Precipitation': 0.31182036432818405, 'Herbicide': 0.3727391817981272}

  [A] Running true / inverted / uniform gamma...

  Running true gamma...
        instance 10/100 ...
        instance 20/100 ...
        instance 30/100 ...
        instance 40/100 ...
        instance 50/100 ...
        instance 60/100 ...
        instance 70/100 ...
        instance 80/100 ...
        instance 90/100 ...
        instance 100/100 ...
    Pearson=0.9843 | Spearman=1.0000 | Cosine=0.9868 | RMSE=0.0511

  Running inverted gamma...
        instance 10/100 ...
        instance 20/100 ...
        instance 30/100 ...
        instance 40/100 ...
        instance 50/100 ...
        instance 60/100 ...
        instance 